# Ticketmaster — San Jose Earthquakes Ticket Scraper

Uses the **Ticketmaster Discovery API v2** to fetch upcoming SJE home events and their ticket price ranges.

Sign up for a free API key at: https://developer.ticketmaster.com

In [11]:
import requests
import pandas as pd
import time

API_KEY = "LyJMmObKVt8HhwrliyabmorlCtma5iUt"  # 替换成你的 Ticketmaster API Key
BASE = "https://app.ticketmaster.com/discovery/v2"
#

## 1. 从本地数据中提取今天以后的 SJE 赛事名称

In [12]:
events_local = pd.read_excel("RESALE_LISTING_TABLES.xlsx", sheet_name="EVENTS")
events_local["START_DATE"] = pd.to_datetime(events_local["START_DATE"])

today = pd.Timestamp.today().normalize()
sje_future = (
    events_local[
        (events_local["GROUP_ID"] == 2583) &
        (events_local["START_DATE"] > today)
    ][["ID", "NAME", "START_DATE"]]
    .sort_values("START_DATE")
    .reset_index(drop=True)
)

print(f"Future SJE events: {len(sje_future)}")
sje_future

Future SJE events: 12


,ID,NAME,START_DATE
0,166161,San Jose Earthquakes vs. Austin FC,2026-04-23 02:30:00
1,166164,San Jose Earthquakes vs. Vancouver Whitecaps FC,2026-05-10 02:30:00
2,166168,San Jose Earthquakes vs. FC Dallas,2026-05-17 02:30:00
3,166172,San Jose Earthquakes vs. Orlando City SC,2026-07-23 02:30:00
4,171058,San Jose Earthquakes vs. LA Galaxy,2026-07-26 02:30:00
5,166178,San Jose Earthquakes vs. St. Louis City SC,2026-08-16 02:30:00
6,166175,San Jose Earthquakes vs. Minnesota United FC,2026-08-23 02:30:00
7,166170,San Jose Earthquakes vs. Houston Dynamo FC,2026-09-13 02:30:00
8,166166,San Jose Earthquakes vs. Portland Timbers,2026-09-27 02:30:00
9,166162,San Jose Earthquakes vs. Nashville SC,2026-10-18 02:30:00


## 2. 通过 Ticketmaster API 搜索 SJE 赛事

In [13]:
def fetch_tm_events(keyword="San Jose Earthquakes", size=20):
    """Search Ticketmaster for SJE events, return raw list."""
    params = {
        "apikey": API_KEY,
        "keyword": keyword,
        "classificationName": "Soccer",
        "size": size,
        "sort": "date,asc",
    }
    r = requests.get(f"{BASE}/events.json", params=params, timeout=10)
    r.raise_for_status()
    data = r.json()

    if "_embedded" not in data:
        print("No events found.")
        return []
    return data["_embedded"]["events"]


def parse_tm_events(events):
    """Parse raw TM event list into a DataFrame."""
    rows = []
    for e in events:
        venue = e.get("_embedded", {}).get("venues", [{}])[0]
        price_ranges = e.get("priceRanges", [])
        rows.append({
            "tm_event_id":  e["id"],
            "name":         e["name"],
            "date":         e["dates"]["start"]["localDate"],
            "time":         e["dates"]["start"].get("localTime"),
            "status":       e["dates"]["status"]["code"],
            "venue":        venue.get("name"),
            "city":         venue.get("city", {}).get("name"),
            "min_price":    price_ranges[0].get("min") if price_ranges else None,
            "max_price":    price_ranges[0].get("max") if price_ranges else None,
            "currency":     price_ranges[0].get("currency") if price_ranges else None,
            "url":          e.get("url"),
        })
    return pd.DataFrame(rows)


raw_events = fetch_tm_events()
df_tm = parse_tm_events(raw_events)
df_tm

HTTPError: 401 Client Error: Unauthorized for url: https://app.ticketmaster.com/discovery/v2/events.json?apikey=LyJMmObKVt8HhwrliyabmorlCtma5iUt&keyword=San+Jose+Earthquakes&classificationName=Soccer&size=20&sort=date%2Casc

In [14]:
# Setting API Key param
params = {'apikey':'LyJMmObKVt8HhwrliyabmorlCtma5iUt'}

# Pulling URIs for all country codes
uri_response = requests.get('https://app.ticketmaster.eu/mfxapi/v2', params=params)

# Extracting Europe URI
uri_response.json()

{'timestamp': 1776262759457,
 'status': 404,
 'error': 'Not Found',
 'path': '/discovery-http/rs/v2'}

In [19]:
import requests

url = "https://app.ticketmaster.com/discovery/v2/events.json"
params = {
    "apikey": "ysAgZSGVFoGtFzXWLA8RsmimFTvBHPkH",
    'keyword':'Concert',
    "countryCode": "US",
    "page": 0
}

response = requests.get(url, params=params)
data = response.json()

In [20]:
data

{'_embedded': {'events': [{'name': 'Curebound Concert for Cures: P!NK',
    'type': 'event',
    'id': 'vvG1IZ_AKnSEae',
    'test': False,
    'url': 'https://www.ticketmaster.com/curebound-concert-for-cures-pnk-san-diego-california-05-15-2026/event/0A006454EACE7242',
    'locale': 'en-us',
    'images': [{'ratio': '16_9',
      'url': 'https://s1.ticketm.net/dam/e/812/454f0963-09bf-4932-9666-bb65d3324812_RECOMENDATION_16_9.jpg',
      'width': 100,
      'height': 56,
      'fallback': False},
     {'ratio': '3_2',
      'url': 'https://s1.ticketm.net/dam/e/812/454f0963-09bf-4932-9666-bb65d3324812_TABLET_LANDSCAPE_3_2.jpg',
      'width': 1024,
      'height': 683,
      'fallback': False},
     {'ratio': '16_9',
      'url': 'https://s1.ticketm.net/dam/e/812/454f0963-09bf-4932-9666-bb65d3324812_RETINA_PORTRAIT_16_9.jpg',
      'width': 640,
      'height': 360,
      'fallback': False},
     {'ratio': '4_3',
      'url': 'https://s1.ticketm.net/dam/e/812/454f0963-09bf-4932-9666-bb65

## 3. 获取每场赛事的详细票价档（Offers）

In [8]:
def fetch_event_offers(tm_event_id):
    """Fetch ticket offers for a single TM event."""
    params = {"apikey": API_KEY}
    r = requests.get(f"{BASE}/events/{tm_event_id}/offers.json", params=params, timeout=10)
    if r.status_code != 200:
        return []
    data = r.json()
    return data.get("offers", [])


def fetch_all_offers(df_events, delay=0.3):
    """Loop through events and collect all offer rows."""
    rows = []
    for _, ev in df_events.iterrows():
        offers = fetch_event_offers(ev["tm_event_id"])
        for o in offers:
            rows.append({
                "tm_event_id":  ev["tm_event_id"],
                "event_name":   ev["name"],
                "event_date":   ev["date"],
                "offer_name":   o.get("name"),
                "price":        o.get("prices", [{}])[0].get("value"),
                "currency":     o.get("prices", [{}])[0].get("currency"),
                "ticket_limit": o.get("ticketLimit"),
                "offer_type":   o.get("offerType"),
            })
        time.sleep(delay)  # 避免触发限流
    return pd.DataFrame(rows)


df_offers = fetch_all_offers(df_tm)
print(f"Total offer rows: {len(df_offers)}")
df_offers.head(20)

Total offer rows: 0


""


## 4. 与本地 Tixr 数据对比：各赛事挂单数量 vs Ticketmaster 价格区间

In [ ]:
listings = pd.read_excel("RESALE_LISTING_TABLES.xlsx", sheet_name="LISTINGS")

# 本地 Tixr 数据：各赛事挂单量
sje_event_ids = sje_future["ID"].tolist()
tixr_counts = (
    listings[listings["EVENT_ID"].isin(sje_event_ids)]
    .groupby("EVENT_ID")
    .size()
    .reset_index(name="tixr_listing_count")
    .merge(sje_future[["ID", "NAME", "START_DATE"]], left_on="EVENT_ID", right_on="ID")
    .drop(columns="ID")
    .sort_values("START_DATE")
)

# TM 数据：价格区间
tm_prices = df_tm[["name", "date", "min_price", "max_price", "url"]].copy()

print("=== Tixr listing counts ===")
display(tixr_counts)

print("\n=== Ticketmaster price ranges ===")
display(tm_prices)

## 5. SeatGeek API — 座位级价格数据

SeatGeek 是二级票务市场，API 免费开放，提供比 Ticketmaster 更详细的价格信息：
- 每个区域（section）的最低价
- 全场最低 / 平均 / 最高价
- 座位数量

注册 Client ID：https://seatgeek.com/account/develop

In [ ]:
import requests
import pandas as pd

SG_CLIENT_ID = "your_seatgeek_client_id"  # 替换成你的 SeatGeek Client ID
SG_BASE = "https://api.seatgeek.com/2"


def sg_search_events(keyword, per_page=10, page=1):
    """按关键词搜索 SeatGeek 活动"""
    r = requests.get(f"{SG_BASE}/events", params={
        "client_id": SG_CLIENT_ID,
        "q": keyword,
        "per_page": per_page,
        "page": page,
    }, timeout=10)
    r.raise_for_status()
    data = r.json()
    rows = []
    for e in data.get("events", []):
        venue = e.get("venue", {})
        stats = e.get("stats", {})
        rows.append({
            "sg_event_id":    e["id"],
            "name":           e["title"],
            "date":           e["datetime_local"][:10],
            "time":           e["datetime_local"][11:],
            "venue":          venue.get("name"),
            "city":           venue.get("city"),
            "state":          venue.get("state"),
            "listing_count":  stats.get("listing_count"),     # 当前在售票数
            "average_price":  stats.get("average_price"),     # 平均价
            "lowest_price":   stats.get("lowest_price"),      # 最低价
            "highest_price":  stats.get("highest_price"),     # 最高价
            "url":            e.get("url"),
        })
    total = data.get("meta", {}).get("total", 0)
    return pd.DataFrame(rows), total


df_sg, total = sg_search_events("San Jose Earthquakes", per_page=10)
print(f"共找到 {total} 个结果")
df_sg

In [ ]:
def sg_get_listings(sg_event_id):
    """获取单个活动的分区座位价格（二级市场挂单）"""
    r = requests.get(f"{SG_BASE}/listings", params={
        "client_id": SG_CLIENT_ID,
        "event_id": sg_event_id,
        "per_page": 100,
    }, timeout=10)
    r.raise_for_status()
    listings = r.json().get("listings", [])
    rows = []
    for l in listings:
        rows.append({
            "sg_event_id": sg_event_id,
            "listing_id":  l.get("id"),
            "section":     l.get("section"),
            "row":         l.get("row"),
            "quantity":    l.get("quantity"),
            "price":       l.get("price"),          # 每张票单价（含手续费）
            "score":       l.get("score"),           # SeatGeek 性价比评分 0~1
        })
    return pd.DataFrame(rows)


# 取第一个活动的分区价格（需要先跑上面的 sg_search_events）
if not df_sg.empty:
    first_id = df_sg.iloc[0]["sg_event_id"]
    first_name = df_sg.iloc[0]["name"]
    df_listings = sg_get_listings(first_id)
    print(f"活动：{first_name}")
    print(f"共 {len(df_listings)} 条挂单\n")
    # 按区域汇总最低价
    if not df_listings.empty:
        summary = (df_listings.groupby("section")["price"]
                   .agg(min_price="min", avg_price="mean", listing_count="count")
                   .sort_values("min_price")
                   .reset_index())
        display(summary)
    else:
        print("暂无挂单数据")
else:
    print("请先运行上方搜索 cell")

In [ ]:
import time

def sg_fetch_all_sje(event_names, delay=0.3):
    """批量拉取所有 SJE 赛事的 SeatGeek 价格统计"""
    rows = []
    for name in event_names:
        df, total = sg_search_events(name, per_page=3)
        if df.empty:
            rows.append({"event_name": name, "sg_event_id": None,
                         "lowest_price": None, "average_price": None,
                         "highest_price": None, "listing_count": None})
        else:
            best = df.iloc[0]
            rows.append({
                "event_name":    name,
                "sg_event_id":   best["sg_event_id"],
                "date":          best["date"],
                "lowest_price":  best["lowest_price"],
                "average_price": best["average_price"],
                "highest_price": best["highest_price"],
                "listing_count": best["listing_count"],
                "sg_url":        best["url"],
            })
        time.sleep(delay)
    return pd.DataFrame(rows)


# 用本地 SJE 赛事名批量查询
sje_names = sje_future["NAME"].tolist()
df_sg_all = sg_fetch_all_sje(sje_names)
df_sg_all

## 6. Selenium — 爬取 Ticketmaster 上所有 San Jose Earthquakes 相关活动

通过模拟浏览器访问搜索页，提取活动名称、日期、场馆、城市、票务链接。

In [39]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import subprocess, time, re
import requests
import pandas as pd

TM_API_KEY = "ysAgZSGVFoGtFzXWLA8RsmimFTvBHPkH"
TM_BASE    = "https://app.ticketmaster.com/discovery/v2"


def build_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    # log_output=subprocess.DEVNULL 屏蔽 macOS MallocStackLogging 噪音警告
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install(), log_output=subprocess.DEVNULL),
        options=options,
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def scroll_to_load_all(driver, pause=2.0, max_scrolls=15):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def parse_event_cards(driver):
    """
    解析所有活动卡片。
    
    日期来源两种情况：
      - 长格式 URL: .../event-name-mm-dd-yyyy/event/ID  → 从 URL 提取
      - 短格式 URL: .../event/Z7r9jZ1A...              → 从父级 div 文本提取
        父级文本格式: "July 22, 2026 | JUL | 22 | Wednesday 07:30 PM | ..."
    """
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-testid='eventList']"))
        )
    except Exception:
        pass

    cards = driver.find_elements(By.CSS_SELECTOR, "[data-testid='event-list-link']")
    rows = []

    for card in cards:
        href = card.get_attribute("href") or ""
        if not href or "/event/" not in href:
            continue

        raw_text = card.text.strip()

        # --- 解析名称 / 城市 / 场馆 ---
        text = re.sub(r"^Find Tickets\s*\|\s*", "", raw_text)
        parts = text.split("|")
        name_location = parts[-1].strip() if parts else text
        comma_split = name_location.split(",")
        name  = comma_split[0].strip()
        city  = comma_split[1].strip() if len(comma_split) > 1 else None
        rest  = comma_split[2].strip() if len(comma_split) > 2 else ""
        state = rest[:2] if rest else None
        venue = rest[2:].strip() if len(rest) > 2 else None

        # --- 提取 event ID ---
        eid_match = re.search(r"/event/([^/?]+)", href)
        event_id  = eid_match.group(1) if eid_match else None

        # --- 提取日期 ---
        # 方法1: 长格式 URL 含 mm-dd-yyyy
        date_match = re.search(r"(\d{2}-\d{2}-\d{4})", href)
        if date_match:
            m, d, y = date_match.group(1).split("-")
            date = f"{y}-{m}-{d}"
            time_str = None
        else:
            # 方法2: 从父级 div 文本（level 2）提取 "Month DD, YYYY ... HH:MM PM"
            date, time_str = None, None
            try:
                grandparent = card.find_element(By.XPATH, "../..")
                gp_text = grandparent.text
                # 日期: "April 22, 2026" 或 "July 22, 2026"
                dm = re.search(
                    r"(January|February|March|April|May|June|July|August|"
                    r"September|October|November|December)\s+(\d{1,2}),\s+(\d{4})",
                    gp_text
                )
                if dm:
                    from datetime import datetime
                    date = datetime.strptime(
                        f"{dm.group(1)} {dm.group(2)} {dm.group(3)}", "%B %d %Y"
                    ).strftime("%Y-%m-%d")
                # 时间: "07:30 PM"
                tm2 = re.search(r"(\d{1,2}:\d{2}\s*[AP]M)", gp_text)
                if tm2:
                    time_str = tm2.group(1)
            except Exception:
                pass

        rows.append({
            "event_id": event_id,
            "name":     name,
            "date":     date,
            "time":     time_str,
            "city":     city,
            "state":    state,
            "venue":    venue,
            "url":      href,
        })
    return rows


print("函数已加载，运行下一个 cell 开始爬取。")

函数已加载，运行下一个 cell 开始爬取。


In [40]:
SEARCH_URL = "https://www.ticketmaster.com/search?q=san+jose+earthquakes"

driver = build_driver(headless=True)
try:
    print(f"打开: {SEARCH_URL}")
    driver.get(SEARCH_URL)
    time.sleep(4)
    print("滚动加载...")
    scroll_to_load_all(driver, pause=2.0, max_scrolls=20)
    print("解析卡片...")
    rows = parse_event_cards(driver)
finally:
    driver.quit()

df_tm_sel = pd.DataFrame(rows)
df_tm_sel = df_tm_sel[df_tm_sel["name"].str.strip() != ""].reset_index(drop=True)
df_tm_sel = df_tm_sel.sort_values("date").reset_index(drop=True)

print(f"\n共爬取到 {len(df_tm_sel)} 场活动，其中有日期: {df_tm_sel['date'].notna().sum()} 场\n")
df_tm_sel

python(15709) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15710) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15711) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15712) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15713) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15714) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15715) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15716) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15717) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15718) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(15719) Malloc

打开: https://www.ticketmaster.com/search?q=san+jose+earthquakes
滚动加载...
解析卡片...

共爬取到 22 场活动，其中有日期: 22 场



,event_id,name,date,time,city,state,venue,url
0,0A006373F363940F,Find Tickets\nLos Angeles Football Club vs. Sa...,2026-04-19,None,Los Angeles,CA,BMO Stadium,https://www.ticketmaster.com/los-angeles-footb...
1,Z7r9jZ1A7roAg,Find Tickets\nSan Jose Earthquakes vs. Austin FC,2026-04-22,07:30 PM,San Jose,CA,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAg
2,Z7r9jZ1A7rov0,Find Tickets\nSt. Louis City SC vs. San Jose E...,2026-04-25,07:30 PM,Saint Louis,MO,Energizer Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rov0
3,09006426D9649A12,Find Tickets\nNetflix Is A Joke Presents: Katt...,2026-05-05,None,California Intuit Dome,None,None,https://www.ticketmaster.com/netflix-is-a-joke...
4,Z7r9jZ1A7roAU,Find Tickets\nSan Jose Earthquakes vs. Vancouv...,2026-05-09,07:30 PM,San Jose,CA,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAU
5,0F006383ED599A47,Find Tickets\nSeattle Sounders FC vs. San Jose...,2026-05-13,None,Seattle,WA,Lumen Field,https://www.ticketmaster.com/seattle-sounders-...
6,Z7r9jZ1A7roAz,Find Tickets\nSan Jose Earthquakes vs. FC Dallas,2026-05-16,07:30 PM,San Jose,CA,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAz
7,Z7r9jZ1A7roZ3,Find Tickets\nPortland Timbers vs. San Jose Ea...,2026-05-23,06:30 PM,Portland,OR,Providence Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roZ3
8,Z7r9jZ1A7rokZ,Find Tickets\nSan Jose Earthquakes vs. Orlando...,2026-07-22,07:30 PM,San Jose,CA,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rokZ
9,Z7r9jZ1A7ro7d,Find Tickets\nFC Cincinnati vs. San Jose Earth...,2026-08-01,07:30 PM,Cincinnati,OH,TQL Stadium,https://www.ticketmaster.com/event/Z7r9jZ1A7ro7d


In [42]:
def tm_api_event_detail(event_id):
    """用 TM Discovery API 补充单场活动详情：日期、票价、票档、出售状态"""
    r = requests.get(
        f"{TM_BASE}/events/{event_id}.json",
        params={"apikey": TM_API_KEY},
        timeout=10,
    )
    if r.status_code != 200:
        return {}
    d = r.json()
    pr = d.get("priceRanges", [])
    sales = d.get("sales", {}).get("public", {})
    return {
        "api_date":       d.get("dates", {}).get("start", {}).get("localDate"),
        "api_time":       d.get("dates", {}).get("start", {}).get("localTime"),
        "sale_status":    d.get("dates", {}).get("status", {}).get("code"),
        "sale_start":     sales.get("startDateTime"),
        "sale_end":       sales.get("endDateTime"),
        "price_min":      pr[0].get("min") if pr else None,
        "price_max":      pr[0].get("max") if pr else None,
        "price_currency": pr[0].get("currency") if pr else None,
        "ticket_limit":   d.get("ticketLimit", {}).get("info"),
        "seatmap_url":    d.get("seatmap", {}).get("staticUrl"),
    }


# --- 1. 用 API 补全日期 + 详情 ---
print("通过 API 补全每场活动详情...")
details = []
for _, row in df_tm_sel.iterrows():
    det = tm_api_event_detail(row["event_id"])
    # 日期优先用 Selenium 爬到的，缺失时用 API 补
    details.append({
        **row.to_dict(),
        "date":        row["date"] or det.get("api_date"),
        "time":        row["time"] or det.get("api_time"),
        "sale_status": det.get("sale_status"),
        "sale_start":  det.get("sale_start"),
        "sale_end":    det.get("sale_end"),
        "price_min":   det.get("price_min"),
        "price_max":   det.get("price_max"),
        "ticket_limit":det.get("ticket_limit"),
        "seatmap_url": det.get("seatmap_url"),
    })
    time.sleep(0.2)

df_detail = pd.DataFrame(details).sort_values("date").reset_index(drop=True)

# --- 2. 与 Tixr 本地赛事对比 ---
# 本地 SJE 赛事
tixr_sje = sje_future[["ID", "NAME", "START_DATE"]].copy()
tixr_sje["START_DATE_str"] = tixr_sje["START_DATE"].dt.strftime("%Y-%m-%d")

# TM 上只保留 SJE 相关
def is_sje(name):
    n = str(name).lower()
    return "earthquake" in n or "san jose" in n

df_sje = df_detail[df_detail["name"].apply(is_sje)].copy()
df_other = df_detail[~df_detail["name"].apply(is_sje)].copy()

# 模糊匹配：提取对手队名做 join
def opponent(name):
    """从 'Team A vs. Team B' 提取对手"""
    m = re.search(r"vs\.?\s*(.+)", name, re.IGNORECASE)
    return m.group(1).strip() if m else name

df_sje["opponent"] = df_sje["name"].apply(opponent)
tixr_sje["opponent"] = tixr_sje["NAME"].apply(opponent)

merged = tixr_sje.merge(
    df_sje[["opponent", "date", "time", "venue", "city", "state",
            "sale_status", "price_min", "price_max", "ticket_limit",
            "seatmap_url", "url"]],
    on="opponent", how="left"
).drop(columns="opponent")

print(f"\nTixr 本地 SJE 赛事: {len(tixr_sje)} 场")
print(f"Ticketmaster 上找到匹配: {merged['url'].notna().sum()} 场")
print(f"Ticketmaster 上未找到:   {merged['url'].isna().sum()} 场\n")

print("=== 对比结果（含 TM 挂票详情）===")
display(merged[["NAME", "START_DATE_str", "sale_status",
                "price_min", "price_max", "ticket_limit", "venue", "url"]])

print("\n=== TM 上其他相关活动（客场赛）===")
display(df_other[["name", "date", "venue", "city", "state", "sale_status", "url"]])

通过 API 补全每场活动详情...

Tixr 本地 SJE 赛事: 12 场
Ticketmaster 上找到匹配: 7 场
Ticketmaster 上未找到:   5 场

=== 对比结果（含 TM 挂票详情）===


,NAME,START_DATE_str,sale_status,price_min,price_max,ticket_limit,venue,url
0,San Jose Earthquakes vs. Austin FC,2026-04-23,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAg
1,San Jose Earthquakes vs. Vancouver Whitecaps FC,2026-05-10,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAU
2,San Jose Earthquakes vs. FC Dallas,2026-05-17,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7roAz
3,San Jose Earthquakes vs. Orlando City SC,2026-07-23,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rokZ
4,San Jose Earthquakes vs. LA Galaxy,2026-07-26,NaN,NaN,NaN,NaN,NaN,NaN
5,San Jose Earthquakes vs. St. Louis City SC,2026-08-16,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rok7
6,San Jose Earthquakes vs. Minnesota United FC,2026-08-23,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rokk
7,San Jose Earthquakes vs. Houston Dynamo FC,2026-09-13,NaN,NaN,NaN,NaN,NaN,NaN
8,San Jose Earthquakes vs. Portland Timbers,2026-09-27,onsale,None,None,None,PayPal Park,https://www.ticketmaster.com/event/Z7r9jZ1A7rok3
9,San Jose Earthquakes vs. Nashville SC,2026-10-18,NaN,NaN,NaN,NaN,NaN,NaN



=== TM 上其他相关活动（客场赛）===


,name,date,venue,city,state,sale_status,url
3,Find Tickets\nNetflix Is A Joke Presents: Katt...,2026-05-05,None,California Intuit Dome,None,None,https://www.ticketmaster.com/netflix-is-a-joke...
17,Find Tickets\nKIDZ BOP LIVE Birthday Concert E...,2026-09-20,None,California Long Beach Amphitheater,None,None,https://www.ticketmaster.com/kidz-bop-live-bir...


## 7. Selenium — 爬取每场活动的座位 Listing 详情

`[data-bdd='listing-card']` 包含每张在售票的 Section、Row、价格、票类型（Primary / Resale）。

In [ ]:
LISTING_COLS = ["event_name", "event_date", "section", "row", "ticket_type", "price", "entry_method"]


def get_page(driver, url, retries=3, wait=6):
    """带重试的页面加载"""
    for attempt in range(retries):
        try:
            driver.get(url)
            time.sleep(wait)
            return True
        except Exception as e:
            print(f"  [重试 {attempt+1}/{retries}] {e.__class__.__name__}")
            time.sleep(3)
    return False


def scrape_event_listings(event_name, event_date, event_url):
    """爬取单场活动的所有 listing（Section / Row / Price）"""
    driver = build_driver(headless=True)
    rows = []
    try:
        if not get_page(driver, event_url, wait=6):
            print("  页面加载失败，跳过")
            return pd.DataFrame(columns=LISTING_COLS)

        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "[data-bdd='quick-picks-list-item-resale']")
                )
            )
        except Exception:
            pass

        try:
            panel = driver.find_element(By.CSS_SELECTOR, "[data-bdd='qp-split-scroll']")
            for _ in range(3):
                driver.execute_script(
                    "arguments[0].scrollTop = arguments[0].scrollHeight", panel
                )
                time.sleep(0.5)
        except Exception:
            pass

        cards = driver.find_elements(
            By.CSS_SELECTOR, "[data-bdd='quick-picks-list-item-resale']"
        )
        print(f"  找到 {len(cards)} 张 listing card")

        for card in cards:
            lines = [l.strip() for l in card.text.split("\n") if l.strip()]
            section, row, ticket_type, price, entry = None, None, None, None, None
            for line in lines:
                if line.startswith("Sec"):
                    parts = line.split("•")
                    section = parts[0].replace("Sec", "").strip()
                    row = parts[1].replace("Row", "").strip() if len(parts) > 1 else None
                elif line.startswith("$"):
                    price = float(line.replace("$", "").replace(",", ""))
                elif "Ticket" in line:
                    ticket_type = line
                elif "Entry" in line or "Delivery" in line:
                    entry = line
            rows.append({
                "event_name":   event_name,
                "event_date":   event_date,
                "section":      section,
                "row":          row,
                "ticket_type":  ticket_type,
                "price":        price,
                "entry_method": entry,
            })
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=LISTING_COLS) if rows else pd.DataFrame(columns=LISTING_COLS)


# 测试：单场活动
test_url  = "https://www.ticketmaster.com/event/Z7r9jZ1A7rokZ"
test_name = "San Jose Earthquakes vs. Orlando City SC"
test_date = "2026-07-22"

print(f"爬取: {test_name}")
df_listings_test = scrape_event_listings(test_name, test_date, test_url)
print(f"共 {len(df_listings_test)} 条\n")
display(df_listings_test)

if not df_listings_test.empty:
    print("\n=== 按 Section 汇总 ===")
    summary = (
        df_listings_test.groupby("section")["price"]
        .agg(count="count", min_price="min", avg_price="mean")
        .sort_values("min_price").reset_index()
    )
    display(summary)

In [ ]:
# 批量爬取 TM 搜索结果里所有 SJE 相关赛事的 listing
# 直接从 df_tm_sel 过滤，不依赖 merged

def is_sje(name):
    n = str(name).lower()
    return "earthquake" in n or "san jose" in n

tm_sje_events = df_tm_sel[df_tm_sel["name"].apply(is_sje)][["name", "date", "url"]].copy()
print(f"准备爬取 {len(tm_sje_events)} 场 SJE 赛事的 listing...\n")

all_listings = []
for _, ev in tm_sje_events.iterrows():
    print(f"→ {ev['name']} ({ev['date']})")
    df_l = scrape_event_listings(ev["url"])
    df_l["event_name"] = ev["name"]
    df_l["event_date"] = ev["date"]
    all_listings.append(df_l)
    time.sleep(1)

df_all_listings = pd.concat(all_listings, ignore_index=True)
print(f"\n全部 listing 共 {len(df_all_listings)} 条")
df_all_listings

# Section 8: Tixr Resale Listing Tables 深度分析

分析 `RESALE_LISTING_TABLES.xlsx` 中的 6 张表，重点关注：
1. 整体挂单量与出售率
2. 各赛事价格分布与溢价
3. 卖家调价行为
4. 活动时间线与票务热度
5. San Jose Earthquakes (GROUP_ID=2583) 专项分析

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# --- 读取全部表 ---
xl = pd.ExcelFile("RESALE_LISTING_TABLES.xlsx")
listings     = xl.parse("LISTINGS")
items        = xl.parse("LISTING_ITEMS")
activities   = xl.parse("LISTING_ACTIVITIES")
act_items    = xl.parse("LISTING_ACTIVITY_ITEMS")
events       = xl.parse("EVENTS")
client_groups = xl.parse("CLIENT_GROUPS")

# 类型整理
items["PRICE"]                  = pd.to_numeric(items["PRICE"], errors="coerce")
items["ORIGINAL_DISPLAY_PRICE"] = pd.to_numeric(items["ORIGINAL_DISPLAY_PRICE"], errors="coerce")
items["PAYOUT_AMOUNT"]          = pd.to_numeric(items["PAYOUT_AMOUNT"], errors="coerce")
items["TIXR_FEE"]               = pd.to_numeric(items.get("TIXR_FEE", pd.Series(dtype=float)), errors="coerce")
act_items["PREVIOUS_PRICE"]     = pd.to_numeric(act_items["PREVIOUS_PRICE"], errors="coerce")
act_items["NEW_PRICE"]          = pd.to_numeric(act_items["NEW_PRICE"], errors="coerce")
events["START_DATE"]            = pd.to_datetime(events["START_DATE"], errors="coerce")
listings["CREATE_DATE"]         = pd.to_datetime(listings["CREATE_DATE"], errors="coerce")
listings["LMD_DATE"]            = pd.to_datetime(listings["LMD_DATE"], errors="coerce")

# SJE 筛选
SJE_GROUP = 2583
sje_events    = events[events["GROUP_ID"] == SJE_GROUP][["ID","NAME","START_DATE"]].copy()
sje_event_ids = sje_events["ID"].tolist()
sje_listings  = listings[listings["EVENT_ID"].isin(sje_event_ids)].copy()
sje_items     = items[items["LISTING_ID"].isin(sje_listings["ID"])].copy()

print(f"全库: {len(listings):,} listings | {len(items):,} items | {len(events):,} events")
print(f"SJE : {len(sje_listings):,} listings | {len(sje_items):,} items | {len(sje_events):,} events")
print(f"\n数据时间范围: {listings['CREATE_DATE'].min().date()} ~ {listings['CREATE_DATE'].max().date()}")

## 8.1 整体挂单状态分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 全库 item 状态
status_all = items["STATUS"].value_counts()
axes[0].pie(status_all.values, labels=status_all.index, autopct="%1.1f%%", startangle=90,
            colors=["#4CAF50","#FFC107","#F44336","#9E9E9E"][:len(status_all)])
axes[0].set_title("全库 LISTING_ITEMS 状态分布", fontsize=12)

# SJE item 状态
status_sje = sje_items["STATUS"].value_counts()
axes[1].pie(status_sje.values, labels=status_sje.index, autopct="%1.1f%%", startangle=90,
            colors=["#4CAF50","#FFC107","#F44336","#9E9E9E"][:len(status_sje)])
axes[1].set_title("SJE LISTING_ITEMS 状态分布", fontsize=12)

plt.tight_layout()
plt.show()

# 数值汇总
print("=== 全库 STATUS 分布 ===")
print(status_all.to_frame("count").assign(pct=lambda df: (df["count"]/df["count"].sum()*100).round(1)))
print("\n=== SJE STATUS 分布 ===")
print(status_sje.to_frame("count").assign(pct=lambda df: (df["count"]/df["count"].sum()*100).round(1)))

## 8.2 SJE 各赛事挂单量 & 出售率

In [ ]:
# 合并 listing → item，关联赛事名称
sje_full = (
    sje_items
    .merge(sje_listings[["ID","EVENT_ID"]], left_on="LISTING_ID", right_on="ID", suffixes=("","_lst"))
    .merge(sje_events[["ID","NAME","START_DATE"]], left_on="EVENT_ID", right_on="ID", suffixes=("","_ev"))
)

# 每场赛事: 总票数、出售数、出售率
event_stats = (
    sje_full.groupby(["NAME","START_DATE"])
    .agg(
        total_items   = ("STATUS","count"),
        sold_items    = ("STATUS", lambda x: (x=="SOLD").sum()),
        active_items  = ("STATUS", lambda x: (x=="ACTIVE").sum()),
        archived_items= ("STATUS", lambda x: (x=="ARCHIVED").sum()),
        avg_price     = ("PRICE","mean"),
        min_price     = ("PRICE","min"),
        max_price     = ("PRICE","max"),
    )
    .reset_index()
    .sort_values("START_DATE")
)
event_stats["sell_through"] = (event_stats["sold_items"] / event_stats["total_items"] * 100).round(1)
# 简短赛事名（去掉 "San Jose Earthquakes vs. " 前缀）
event_stats["short_name"] = event_stats["NAME"].str.replace(
    r"San Jose Earthquakes vs\.?\s*", "", regex=True
)

fig, ax1 = plt.subplots(figsize=(14, 6))
x = range(len(event_stats))
bars = ax1.bar(x, event_stats["total_items"], color="#90CAF9", label="Total items")
ax1.bar(x, event_stats["sold_items"], color="#4CAF50", label="SOLD")
ax1.bar(x, event_stats["active_items"], bottom=event_stats["sold_items"], color="#FFC107", label="ACTIVE")
ax1.set_xticks(list(x))
ax1.set_xticklabels(event_stats["short_name"], rotation=35, ha="right", fontsize=8)
ax1.set_ylabel("Ticket count")
ax1.set_title("SJE 各赛事 Resale Listing Items（已售 vs 在售 vs 下架）")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(list(x), event_stats["sell_through"], "o--", color="#E53935", linewidth=2, label="Sell-through %")
ax2.set_ylabel("Sell-through rate (%)")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

print("\n=== 各赛事汇总 ===")
display(event_stats[["short_name","START_DATE","total_items","sold_items",
                      "sell_through","avg_price","min_price","max_price"]]
        .rename(columns={"short_name":"对手","START_DATE":"日期","total_items":"总票数",
                          "sold_items":"已售","sell_through":"出售率%",
                          "avg_price":"均价","min_price":"最低价","max_price":"最高价"})
        .assign(均价=lambda d: d["均价"].round(2))
        .to_string(index=False))

## 8.3 价格分布 & 溢价分析（售价 vs 原价）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: SJE 各赛事 平均售价 箱线图（按赛事）
price_by_event = [
    grp["PRICE"].dropna().values
    for _, grp in sje_full.groupby("NAME")
]
short_names = [
    n.replace("San Jose Earthquakes vs. ", "").replace("San Jose Earthquakes vs ", "")
    for n in sje_full["NAME"].unique()
]
# 按 START_DATE 排序
name_date = sje_full.groupby("NAME")["START_DATE"].first().reset_index().sort_values("START_DATE")
ordered_names = name_date["NAME"].tolist()
ordered_short = [n.replace("San Jose Earthquakes vs. ","").replace("San Jose Earthquakes vs ","") for n in ordered_names]
ordered_prices = [sje_full[sje_full["NAME"]==n]["PRICE"].dropna().values for n in ordered_names]

bp = axes[0].boxplot(ordered_prices, vert=True, patch_artist=True,
                      boxprops=dict(facecolor="#90CAF9", alpha=0.7),
                      medianprops=dict(color="red", linewidth=2),
                      flierprops=dict(marker=".", markersize=3, alpha=0.3))
axes[0].set_xticklabels(ordered_short, rotation=40, ha="right", fontsize=7)
axes[0].set_ylabel("Price (USD)")
axes[0].set_title("SJE 各赛事 Resale 价格分布")

# 右图: 溢价率直方图（(售价-原价)/原价）
price_comp = sje_full[
    sje_full["PRICE"].notna() &
    sje_full["ORIGINAL_DISPLAY_PRICE"].notna() &
    (sje_full["ORIGINAL_DISPLAY_PRICE"] > 0)
].copy()
price_comp["markup_pct"] = (
    (price_comp["PRICE"] - price_comp["ORIGINAL_DISPLAY_PRICE"])
    / price_comp["ORIGINAL_DISPLAY_PRICE"] * 100
)
markup_clipped = price_comp["markup_pct"].clip(-50, 150)
axes[1].hist(markup_clipped, bins=40, color="#FF8A65", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1.5, linestyle="--", label="面值线")
axes[1].axvline(price_comp["markup_pct"].median(), color="red", linewidth=1.5,
                label=f"中位溢价: {price_comp['markup_pct'].median():.1f}%")
axes[1].set_xlabel("溢价率 % (售价 vs 原价)")
axes[1].set_ylabel("Listing count")
axes[1].set_title("Resale 溢价率分布（SJE）")
axes[1].legend()

plt.tight_layout()
plt.show()

# 统计摘要
above_face = (price_comp["markup_pct"] > 0).sum()
below_face = (price_comp["markup_pct"] <= 0).sum()
print(f"有原价记录: {len(price_comp):,} items")
print(f"高于面值: {above_face:,} ({above_face/len(price_comp)*100:.1f}%)")
print(f"低于/等于面值: {below_face:,} ({below_face/len(price_comp)*100:.1f}%)")
print(f"中位溢价率: {price_comp['markup_pct'].median():.1f}%")
print(f"平均溢价率: {price_comp['markup_pct'].mean():.1f}%")

## 8.4 卖家调价行为分析

利用 LISTING_ACTIVITY_ITEMS 追踪每笔 listing 的历史调价，分析卖家定价策略。

In [ ]:
# 关联 SJE 调价记录
sje_item_ids = sje_items["ID"].tolist()

# LISTING_ACTIVITY_ITEMS 通过 LISTING_ITEM_ID 关联到 sje_items
price_changes = act_items[
    act_items["LISTING_ITEM_ID"].isin(sje_item_ids) &
    act_items["PREVIOUS_PRICE"].notna() &
    act_items["NEW_PRICE"].notna()
].copy()
price_changes["price_delta"] = price_changes["NEW_PRICE"] - price_changes["PREVIOUS_PRICE"]
price_changes["pct_change"]  = price_changes["price_delta"] / price_changes["PREVIOUS_PRICE"] * 100

increases = (price_changes["price_delta"] > 0).sum()
decreases = (price_changes["price_delta"] < 0).sum()
no_change = (price_changes["price_delta"] == 0).sum()
total_pc  = len(price_changes)

print(f"SJE 调价记录共: {total_pc:,} 条")
print(f"  涨价: {increases:,} ({increases/total_pc*100:.1f}%)")
print(f"  降价: {decreases:,} ({decreases/total_pc*100:.1f}%)")
print(f"  不变: {no_change:,} ({no_change/total_pc*100:.1f}%)")
print(f"\n平均调价幅度: ${price_changes['price_delta'].mean():.2f}")
print(f"平均降价幅度: ${price_changes[price_changes['price_delta']<0]['price_delta'].mean():.2f}")
print(f"平均涨价幅度: ${price_changes[price_changes['price_delta']>0]['price_delta'].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左: 调价幅度分布
delta_clipped = price_changes["price_delta"].clip(-80, 80)
axes[0].hist(delta_clipped, bins=50, color="#7986CB", edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1.5, linestyle="--")
axes[0].axvline(price_changes["price_delta"].median(), color="red", linewidth=1.5,
                label=f"中位调价: ${price_changes['price_delta'].median():.1f}")
axes[0].set_xlabel("价格变动 ($)")
axes[0].set_ylabel("次数")
axes[0].set_title("卖家调价幅度分布（SJE）")
axes[0].legend()

# 右: 调价前后散点
sample = price_changes.sample(min(1000, len(price_changes)), random_state=42)
colors = sample["price_delta"].apply(lambda d: "#4CAF50" if d < 0 else "#F44336")
axes[1].scatter(sample["PREVIOUS_PRICE"], sample["NEW_PRICE"],
                c=colors, alpha=0.4, s=15)
lim = max(sample["PREVIOUS_PRICE"].max(), sample["NEW_PRICE"].max()) * 1.05
axes[1].plot([0, lim], [0, lim], "k--", linewidth=1, label="y=x (不变)")
axes[1].set_xlabel("调价前 ($)")
axes[1].set_ylabel("调价后 ($)")
axes[1].set_title("调价前 vs 调价后（绿=降价 / 红=涨价）")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8.5 活动时间线：挂单量随时间变化

In [ ]:
# 关联 listing CREATE_DATE 到 SJE
sje_listings_dated = sje_listings[sje_listings["CREATE_DATE"].notna()].copy()
sje_listings_dated = sje_listings_dated.merge(
    sje_events[["ID","NAME","START_DATE"]], left_on="EVENT_ID", right_on="ID"
)
sje_listings_dated["days_before"] = (
    sje_listings_dated["START_DATE"] - sje_listings_dated["CREATE_DATE"]
).dt.days

# 聚合：每天创建的 listing 数量
daily = (
    sje_listings_dated
    .groupby(sje_listings_dated["CREATE_DATE"].dt.date)
    .size()
    .reset_index(name="new_listings")
)
daily["date"] = pd.to_datetime(daily["CREATE_DATE"])
daily = daily.sort_values("date")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 上图: 每日新增挂单
axes[0].fill_between(daily["date"], daily["new_listings"], alpha=0.6, color="#42A5F5")
axes[0].plot(daily["date"], daily["new_listings"], color="#1565C0", linewidth=0.8)
axes[0].set_title("SJE Resale 每日新增挂单量")
axes[0].set_ylabel("新增 listings")

# 在图上标记各赛事日期
for _, ev in sje_events.sort_values("START_DATE").iterrows():
    if pd.notna(ev["START_DATE"]):
        axes[0].axvline(ev["START_DATE"], color="red", alpha=0.3, linewidth=1, linestyle=":")

# 下图: 提前多少天挂单分布
valid_days = sje_listings_dated[
    sje_listings_dated["days_before"].between(0, 365)
]["days_before"]
axes[1].hist(valid_days, bins=60, color="#AB47BC", edgecolor="white", alpha=0.8)
axes[1].axvline(valid_days.median(), color="red", linewidth=1.5,
                label=f"中位: {valid_days.median():.0f} 天前")
axes[1].set_xlabel("距离赛事天数（天前挂单）")
axes[1].set_ylabel("listing 数量")
axes[1].set_title("卖家提前多少天挂单（SJE）")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"平均提前 {valid_days.mean():.0f} 天挂单")
print(f"中位提前 {valid_days.median():.0f} 天挂单")
print(f"最大提前 {valid_days.max()} 天 | 最小 {valid_days.min()} 天")

## 8.6 Delivery Type & Activity Type 分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左: SJE listing 的交付方式
if "DELIVERY_TYPE" in sje_listings.columns:
    deliv = sje_listings["DELIVERY_TYPE"].value_counts()
    axes[0].bar(deliv.index, deliv.values, color="#80DEEA")
    axes[0].set_title("SJE Resale Listings — 交付方式")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=30)
else:
    axes[0].text(0.5, 0.5, "DELIVERY_TYPE 列不存在", ha="center", va="center")

# 右: SJE 活动类型分布
sje_listing_ids = sje_listings["ID"].tolist()
sje_activities = activities[activities["LISTING_ID"].isin(sje_listing_ids)]
act_types = sje_activities["TYPE"].value_counts()
colors_act = ["#4CAF50","#F44336","#FFC107","#42A5F5","#AB47BC","#FF7043"][:len(act_types)]
axes[1].bar(act_types.index, act_types.values, color=colors_act)
axes[1].set_title("SJE Listing Activities — 操作类型")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print("=== SJE Activity Types ===")
print(act_types.to_frame("count").assign(
    pct=lambda df: (df["count"]/df["count"].sum()*100).round(1)
))

## 8.7 综合汇总：各赛事完整指标对比表

In [ ]:
# 调价次数 per event
price_change_per_event = (
    price_changes
    .merge(sje_items[["ID","LISTING_ID"]], left_on="LISTING_ITEM_ID", right_on="ID")
    .merge(sje_listings[["ID","EVENT_ID"]], left_on="LISTING_ID", right_on="ID")
    .merge(sje_events[["ID","NAME"]], left_on="EVENT_ID", right_on="ID")
    .groupby("NAME")
    .agg(price_change_count=("price_delta","count"),
         avg_delta=("price_delta","mean"))
    .reset_index()
)

# 合并所有指标
summary_full = event_stats.merge(price_change_per_event, on="NAME", how="left")
summary_full["avg_delta"] = summary_full["avg_delta"].round(2)
summary_full["avg_price"] = summary_full["avg_price"].round(2)

display_cols = {
    "short_name": "对手",
    "START_DATE": "赛事日期",
    "total_items": "总票数",
    "sold_items": "已售",
    "active_items": "在售",
    "sell_through": "出售率%",
    "avg_price": "均价($)",
    "min_price": "最低价($)",
    "max_price": "最高价($)",
    "price_change_count": "调价次数",
    "avg_delta": "平均调价($)",
}
out = summary_full[list(display_cols.keys())].rename(columns=display_cols)
out["赛事日期"] = out["赛事日期"].dt.strftime("%Y-%m-%d")
out = out.fillna(0)

print("=== SJE 各赛事 Resale 综合指标 ===\n")
print(out.to_string(index=False))

# 高亮最热场次（出售率最高）
top_sell = out.loc[out["出售率%"].idxmax()]
top_vol  = out.loc[out["总票数"].idxmax()]
top_price = out.loc[out["均价($)"].idxmax()]
print(f"\n最高出售率: {top_sell['对手']} ({top_sell['出售率%']}%)")
print(f"挂单量最多: {top_vol['对手']} ({int(top_vol['总票数'])} 张)")
print(f"均价最高:   {top_price['对手']} (${top_price['均价($)']})")